# "THE POUR IS RIGHT" - Wine Pricer Capstone

# Week 8: from a model to a product

## Order of play

DAY 1: Modal and the SpecialistAgent  
DAY 2: RAG with Qdrant, FrontierAgent, Ensemble  
DAY 3: ScannerAgent, MessagingAgent  
DAY 4: The planning agent  
DAY 5: The user interface  

## DAY 1: serve the fine-tuned model

Last week's model only exists inside a Colab session. Today we put it behind a service that any program can call, using Modal. This follows `week8/day1.ipynb`.

Modal runs your Python functions on its computers, including ones with GPUs, and charges by the second. You describe the machine you need in code, and Modal builds it.

## Before you start

1. Create an account at https://modal.com
2. In a terminal at the repo root, run `uv run modal token new` and approve it in the browser.
3. In the Modal dashboard, go to Secrets, create a new secret of type Hugging Face, name it exactly `huggingface-secret`, and set the key `HF_TOKEN` to your Hugging Face token. Modal needs it to download your private fine-tuned model.

## Costs

Modal gives free monthly credits. A T4 GPU costs well under a dollar an hour, and our service switches itself off two minutes after its last call, so today's work uses a few cents of those credits.

In [ ]:
import sys
import logging
import modal
from dotenv import load_dotenv

load_dotenv(override=True)

## Hello, Modal

`hello.py` holds one small function that looks up where in the world it is running. Open the file: the function is ordinary Python with one extra line above it, `@app.function(image=image)`. That line is all it takes to make it runnable on Modal.

First we run it locally, on this laptop.

In [ ]:
from hello import app, hello

In [ ]:
with app.run():
    reply = hello.local()
reply

Now the same function, with `.remote()` instead of `.local()`. Modal builds a small machine in its cloud, runs the function there, and sends back the answer. The first run takes a little longer while it builds the machine.

In [ ]:
with app.run():
    reply = hello.remote()
reply

The two answers should name different places: yours, and one of Modal's data centres.

## Deploy the wine pricer

`pricer_service.py` describes our service. Open it and look for these parts:

- **The image**: a Linux machine with the same library versions we trained with.
- **`gpu="T4"`**: the same kind of GPU as on Colab.
- **The secret**: gives the machine your Hugging Face token.
- **The volume**: a disk that survives between runs. The 8 GB base model is downloaded to it once and reused, instead of being downloaded on every start.
- **`@modal.enter()`**: runs once when a machine starts. It loads the base model in 4 bits and attaches your adapter, exactly as the Colab notebooks did.
- **`@modal.method()`**: the `price` function that callers use. It builds the same prompt the model was trained on, ending in "Price is $", and returns the number.

`modal deploy` uploads this and gives it a permanent name, `wine-pricer-service`. Deploying is free; you are only charged while a machine is running.

In [ ]:
!{sys.executable} -m modal deploy -m pricer_service

## Call it

`modal.Cls.from_name` finds the deployed service by name. Nothing is running yet at this point.

The first call is slow, perhaps 10 minutes: Modal builds the image and downloads the base model to the volume. After that, a cold start takes about a minute and a warm call a couple of seconds. You can watch it happen at https://modal.com/apps

In [ ]:
Pricer = modal.Cls.from_name("wine-pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
wine = """Wine: Calcareous – Calcareous Meritage, 2010
Style: Red, full-bodied
Origin: Paso Robles, California, US
Points: 90
Tasting: Rich aromas of blackberries, cherries, currants and cocoa with a toasty oak note; tight tannins and a clean, elegant structure.
Verdict: The reviewer says it needs time to soften and mellow, recommending to drink it until late 2010."""

pricer.price.remote(wine)

That is the first wine in our test set. Its real price is \$36, and in Colab the fine-tuned model said \$45. The service should give the same answer, because it is the same model with the same prompt.

Run the next cell straight away to see how fast a warm call is.

In [ ]:
pricer.price.remote(wine)

## Optional: keeping it warm

The service goes to sleep two minutes after its last call, and the next call then waits about a minute for it to wake.

To keep it awake for 20 minutes at a time, run the cell below. To keep it awake permanently, change `MIN_CONTAINERS = 0` to `1` in `pricer_service.py` and deploy again, but that uses credits continuously, so only do it deliberately.

In [ ]:
# pricer.update_autoscaler(scaledown_window=1200)   # stay warm for 20 minutes after the last call
# pricer.update_autoscaler(scaledown_window=120)    # back to 2 minutes

## The Preprocessor

Our model was trained on summaries in one fixed format. A wine described any other way has to be rewritten into that format first, or the model is being asked about something it has never seen.

The `Preprocessor` in `agents/preprocessor.py` does that with the same model and the same prompt that wrote our training summaries on day 2 of week 6 (`gpt-oss-20b` on Groq). Each call costs a small fraction of a cent.

In [ ]:
from agents.preprocessor import Preprocessor

preprocessor = Preprocessor()
text = preprocessor.preprocess("Chateau Ste. Michelle 2015 Cold Creek Vineyard Cabernet Sauvignon from Columbia Valley, Washington. 91 points. Dark cherry, cocoa and firm tannins; needs a few years in the cellar.")
print(text)

In [ ]:
pricer.price.remote(text)

## The SpecialistAgent

An agent, in Ed's framework, is a small class with one job and a coloured log line so you can see who is doing what. The `SpecialistAgent` in `agents/specialist_agent.py` has the job "ask the fine-tuned model for a price". It is the first of several agents that we will combine later in the week.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.specialist_agent import SpecialistAgent

agent = SpecialistAgent()
agent.price(text)